# Import Utilities

In [ ]:
import google.generativeai as genai
import sys

sys.path.append('..')  # Add the parent directory of 'Stage_1-LLM_Ratings' to the Python path
from Utils.llm_evaluation_utils import *

api_key = os.environ.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)

MODEL = 'gemini-2.5-pro'
MAX_TOKENS = 5000
TEMPERATURE = 1
SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE # or FEEDBACK_SYSTEM_MESSAGE

NUM_TRIALS = 1

safety_settings = [
    {
        'category': 'HARM_CATEGORY_DANGEROUS',
        'threshold': 'BLOCK_NONE',
    },
    {
        'category': 'HARM_CATEGORY_HARASSMENT',
        'threshold': 'BLOCK_NONE',
    },
    {
        'category': 'HARM_CATEGORY_HATE_SPEECH',
        'threshold': 'BLOCK_NONE',
    },
    {
        'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT',
        'threshold': 'BLOCK_NONE',
    },
    {
        'category': 'HARM_CATEGORY_DANGEROUS_CONTENT',
        'threshold': 'BLOCK_NONE',
    },
]

generation_config = {
  'temperature': TEMPERATURE,
#   'top_p': 1,
#   'top_k': 8,
  'max_output_tokens': MAX_TOKENS,
}

model = genai.GenerativeModel(model_name=MODEL,
                              safety_settings=safety_settings,
                              system_instruction=SYSTEM_MESSAGE,
                              generation_config=generation_config)

c:\Users\mahmo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_dir = '../../Dataset/cleaned_dataset'
uicrit_data_file = os.path.join(dataset_dir, 'uicrit_notna_deduped.parquet')
base64_screens_file = os.path.join(dataset_dir, 'base64_screens.parquet')
base64_screens_labeled_file = os.path.join(dataset_dir, 'base64_screens_labeled.parquet')

SELECTED_TASKS = '1000_screens-few_shots'

jsonl_responses_file = f'./gemeni_2_5_pro-responses-{SELECTED_TASKS}.jsonl'

all_results_dir = f'../Results/{SELECTED_TASKS}'
model_results_file = f'gemeni_2_5_pro-ratings-{SELECTED_TASKS}.parquet'

## Defined Functions

In [3]:
def prepare_errors_dict(dir, model_name):
    '''Load and prepare a dictionary of IDs and question numbers with Blocked errors from a JSON file.'''
    if '2.0' in model_name:
        errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_0.json'
    elif '2.5' in model_name:
        errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_5.json'
        
    file_path = os.path.join(dir, errors_file_name)
    try:
        with open(file_path, 'r') as f:
            ids_questions_with_error = json.load(f)
        # Convert the lists to sets
        ids_questions_with_error = {key: set(value) for key, value in ids_questions_with_error.items()}
    
    except FileNotFoundError:
        ids_questions_with_error = {}
    
    return ids_questions_with_error

# Load Data

In [4]:
ids_questions_with_error = prepare_errors_dict('.', MODEL)

base64_screens_labeled_df = pd.read_parquet(base64_screens_labeled_file)
# base64_screens_df = pd.read_parquet(base64_screens_file).set_index('screen_id')

few_shot_samples_df = pd.read_parquet('../few_shot_samples/few_shot_samples_df.parquet')

responses_df = load_responses_df(
    responses_dir=all_results_dir,
    results_file_name=model_results_file,
    uicrit_df=uicrit_data_file,
    num_trials=NUM_TRIALS
)
print('responses_df shape:', responses_df.shape)
responses_df.head(2)

responses_df shape: (997, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,8,3,3,5,8
1,28_T01,28,Enter details to Sing In to Scotiabank.,7,5,5,9,9


Getting the First Task of 50 Screens

In [5]:
# Get first screen_task_id per screen_id (including those with NaNs)
first_tasks = responses_df.drop_duplicates(subset='screen_id', keep='first')

# Get the corresponding screen_task_ids
screen_task_ids = first_tasks['screen_task_id'].unique()

# Filter the full responses_df to keep all rows matching those screen_task_ids
screen_responses_df = responses_df[responses_df['screen_task_id'].isin(screen_task_ids)].copy()
screen_responses_df = screen_responses_df.reset_index(drop=True)

del responses_df
responses_df = screen_responses_df
print('responses_df shape:', responses_df.shape)
responses_df.iloc[9:11]

responses_df shape: (1000, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
9,445_T01,445,Expand the functionality by adding a new feat...,None,None,None,None,None
10,640_T01,640,Choose a station on World FM Radio.,None,None,None,None,None


## !!Temporary

In [7]:
# keep only the first screen_task_id of each screen_id
responses_df = responses_df.drop_duplicates(subset=['screen_id'], keep='first')
# keep first five rows for testing
responses_df = responses_df.head(1)
responses_df.reset_index(drop=True, inplace=True)
responses_df.head(5)

,screen_task_id,screen_id,task,trial,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,1,None,None,None,None,None


# Get Gemini Responses

In [ ]:
def save_json_response(response, screen_id, screen_task_id, trial, metric, jsonl_responses_file):
    # If `response` is a JSON string, parse it
    if isinstance(response, str):
        # Extract the first {...} block using regex
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            json_str = match.group(0)
            try:
                response = json.loads(json_str)
            except json.JSONDecodeError:
                print("Error decoding response JSON")
                return
        else:
            print("No JSON object found in response")
            return

    # Final record to save
    response_data = {
        'screen_id': screen_id,
        'screen_task_id': screen_task_id,
        'trial': trial,
        'metric': metric,
        **response  # Flatten the response into the top-level
    }

    # Save to JSONL
    with open(jsonl_responses_file, 'a') as f:
        f.write(json.dumps(response_data) + '\n')
        
def generate_content_with_backoff(model, contents, max_retries=3, base_delay=2):
    '''
    Calls model.generate_content() with exponential backoff on rate limit errors.
  
    Args:
        model: The model object used for content generation.
        contents: The prompt string or content list
        max_retries: Maximum number of retries in case of rate limit errors.
        base_delay: Base delay (in seconds) for exponential backoff.
  
    Returns:
        The response object from model.generate_content() on successful generation,
        or None if all retries fail.
        Raise an error if error occured other than error code 429 
    '''
    
    for attempt in range(1, max_retries + 1):
        try:
            response = model.generate_content(contents)
            if response.text:
                return response
        except Exception as error:
            if getattr(error, 'code', None) == 429:
                print(f'Rate limit exceeded. Attempt {attempt}/{max_retries}...')
                delay = base_delay * 2 ** (attempt - 1)  # Exponential backoff calculation
                time.sleep(delay)
            else:
                raise error # Raise the error for handling in the outer loop
            
    print(f'Failed to generate content after {max_retries} retries.')
    return None

# def create_content(prompt: str, samples_df: pd.DataFrame, base64_string: str) -> list:
#     """Return a list containing a text prompt and a base64-encoded JPEG image."""
#     return [
#         prompt,
#         {
#             "mime_type": "image/jpeg",
#             "data": base64_string
#         }
#     ]

def create_few_shot_content(prompt: str, samples_df: pd.DataFrame, base64_string: str) -> list:
    """Return a list containing a text prompt and a base64-encoded JPEG image."""
    content = [prompt]
    for _, row in samples_df.iterrows():
        content.append({"mime_type": "image/png", "data": row['base64_screen']})
    content.append({"mime_type": "image/png", "data": base64_string})
    return content


In [ ]:
# Calculate the delay based on your rate limit
requests_limit_per_minute = 150
base_delay = 60.0 / requests_limit_per_minute

incomplete_rows = responses_df[responses_df[EVALUATION_MAIN_ASPECTS].isnull().any(axis=1)]

for index, row in incomplete_rows.iterrows():
    screen_id = row['screen_id']
    # skip if screen_id is in column screen_id in few_shot_samples_df 
    if screen_id in few_shot_samples_df['screen_id'].values:
        # drop that row from responses_df
        responses_df = responses_df[responses_df['screen_id'] != screen_id]
        continue
    screen_task_id = row['screen_task_id']
    base64_string = base64_screens_labeled_df.loc[screen_id, 'base64_screen']
    # trial = row['trial']
    # if trial == 1:
    print(f"Processing: index={index} | screen_task_id={screen_task_id}")
    
    for aspect in EVALUATION_MAIN_ASPECTS:
        if pd.notnull(row[aspect]):  # Skip if already filled
            continue
        prompt = build_rating_prompt(row['task'], GUIDELINES, evaluate=aspect, prompting_type='few-shot', samples=few_shot_samples_df)
        contents = create_few_shot_content(prompt, few_shot_samples_df, base64_string)
                
        try:
            # response = model.generate_content(contents)
            response = generate_content_with_backoff(model, contents, max_retries=3, base_delay=base_delay)
            if response:
                update_rating_in_df(responses_df, screen_task_id, trial=None, metric=aspect, response=response.text)
                save_json_response(response.text, screen_id, screen_task_id, trial=None, metric=aspect, jsonl_responses_file=jsonl_responses_file)
            else:
                raise Exception("Could not update responses_df; response:", response)
            
        except Exception as e:
            print(f'Error with {screen_task_id}: {e}')
            if 'candidate.safety_ratings' in str(e) or 'response.prompt_feedback' in str(e):    # BlockedPromptException
                if screen_task_id not in ids_questions_with_error:
                    ids_questions_with_error[screen_task_id] = set()  # Initialize the set if it's the first occurrence of the screen_task_id
                # ids_questions_with_error[screen_task_id].add(trial)
                continue

        time.sleep(base_delay)

In [ ]:
response

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "```json\n{\"aesthetics_rating\": 4}\n```"
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0,
          "safety_ratings": [
            {
              "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
              "probability": "NEGLIGIBLE"
            },
            {
              "category": "HARM_CATEGORY_HATE_SPEECH",
              "probability": "NEGLIGIBLE"
            },
            {
              "category": "HARM_CATEGORY_HARASSMENT",
              "probability": "NEGLIGIBLE"
            },
            {
              "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
              "probability": "NEGLIGIBLE"
            }
          ]
        }
      ],
      "usage_metadata"

# Explore Results

In [ ]:
display_from_index = 0
index_of_q1 = responses_df.columns.get_loc("Q1")

responses_df.iloc[display_from_index:, index_of_q1:index_of_q1+15].head()

In [ ]:
print('Number of errors: ', len(ids_questions_with_error))
ids_questions_with_error

In [ ]:
columns_with_none = (responses_df.isna() | (responses_df == '')).sum()
columns_with_none

In [5]:
rows_with_none = responses_df[responses_df.isna().any(axis=1)]
rows_with_none

,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating


## Store Errors to JSON File

In [12]:
def set_encoder(obj):
    if isinstance(obj, set):
        return list(obj)        # convert sets to lists
    raise TypeError('Object of type set is not JSON serializable')

# save ids and questions number that encountered errors
if '2.0' in MODEL:
    errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_0.json'
elif '2.5' in MODEL:
    errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_5.json'
with open(errors_file_name, 'w') as f:
    json.dump(ids_questions_with_error, f, default=set_encoder)

# Store Results

In [56]:
os.makedirs(all_results_dir, exist_ok=True) # Ensure the directory exists

parquet_output_file = os.path.join(all_results_dir, model_results_file)

responses_df.to_parquet(parquet_output_file, index=False)